# Notes for development
- Gather historic stock data with yfinance
- Gather stock indicator data
  - SMA/EMA (multiple periods: 10, 20, 50)
  - MACD (line, signal, hist)
  - RSI (14)
  - Bollinger Bands (%B, bandwidth)
  - OBV
  - ATR
  - Stochastic or MFI (pick one)
  - ADX
  - Lagged returns/prices (1–5 days)
  - Volume change or VWAP deviation (if relevant)
- Run data through Time-Series Regression model
- Run data through Neural Network based model
- Gather Financial Article data and feed it into a sentiment analysis language model
  

In [ ]:
!pip install pandas_ta

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 751.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 14.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.43.0
    Uninstalling llvmlite-0.43.0:
      Successfully uninstalled llvmlite-0.43.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninsta

In [ ]:
# Basic DS/ML Libraries
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

# Data Gathering Libraries
import yfinance as yf
import pandas_ta as ta

# Scikit-Learn imports
from sklearn.preprocessing import MinMaxScaler

# Pytorch Imports
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
def get_stock_data(ticker='AAPL', period='5y', interval='1d'):
  try:
    stock = yf.Ticker(ticker)
    df = stock.history(period, interval)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    df.index = pd.to_datetime(df.index)
    return df
  except Exception as e:
    print(f"Error fetching data: {e}")
    return None

In [ ]:
def get_indicator_data(df):
  df['SMA_10'] = df.ta.sma(length=10)
  df['SMA_20'] = df.ta.sma(length=20)
  df['SMA_50'] = df.ta.sma(length=50)

  df['EMA_10'] = df.ta.ema(length=10)
  df['EMA_20'] = df.ta.ema(length=20)

  macd = df.ta.macd()
  df = df.join(macd)

  df['RSI_14'] = df.ta.rsi(length=14)

  bb = df.ta.bbands()
  df = df.join(bb)

  df['OBV'] = df.ta.obv()

  df['ATR_14'] = df.ta.atr(length=14)

  stoch = df.ta.stoch()
  df = df.join(stoch)

  df['MFI_14'] = df.ta.mfi(length=14)

  adx = df.ta.adx()
  df = df.join(adx)

  for lag in [1,2,3,4,5]:
      df[f'Close_lag_{lag}']  = df['Close'].shift(lag)
      df[f'Return_lag_{lag}'] = df['Close'].pct_change(periods=lag).shift(1)  # previous lag returns

  df['Volume_change'] = df['Volume'].pct_change()

  df['VWAP'] = df.ta.vwap(high='High', low='Low', close='Close', volume='Volume')
  df['VWAP_dev'] = (df['Close'] - df['VWAP']) / df['VWAP']

  return df

In [ ]:
df = get_stock_data()
df = get_indicator_data(df)
df_clean = df.copy().iloc[49:]
df_clean['Daily_change'] = df['Close'] - df['Open']
df_clean.head()

,Open,High,Low,Close,Volume,SMA_10,SMA_20,SMA_50,EMA_10,EMA_20,...,Close_lag_3,Return_lag_3,Close_lag_4,Return_lag_4,Close_lag_5,Return_lag_5,Volume_change,VWAP,VWAP_dev,Daily_change
Date,,,,,,,,,,,,,,,,,,,,,
2021-04-26 00:00:00-04:00,131.323815,131.547830,130.086836,131.216675,66905100,130.276778,126.551245,123.047588,129.495147,127.400665,...,130.028412,0.009090,129.648575,-0.003856,131.333557,0.001192,-0.149412,130.950447,0.002033,-0.107140
2021-04-27 00:00:00-04:00,131.499121,131.888728,130.622531,130.895248,66015800,130.272881,127.184341,123.028497,129.749711,127.733483,...,128.508987,0.009138,130.028412,0.012095,129.648575,-0.000890,-0.013292,131.135502,-0.001832,-0.603872
2021-04-28 00:00:00-04:00,130.817366,131.508910,129.619356,130.106354,107760100,130.423854,127.850556,123.036095,129.814555,127.959471,...,130.827072,0.018569,128.508987,0.006667,130.028412,0.009616,0.632338,130.411540,-0.002340,-0.711013
2021-04-29 00:00:00-04:00,132.921194,133.505598,129.005727,130.008942,151101000,130.324507,128.402325,123.087522,129.849898,128.154658,...,131.216675,-0.005509,130.827072,0.012430,128.508987,0.000599,0.402198,130.840089,-0.006352,-2.912252
2021-04-30 00:00:00-04:00,128.353175,130.086886,127.661646,128.041504,109839500,130.061533,128.814328,123.121613,129.521099,128.143882,...,130.895248,-0.009204,131.216675,-0.006254,130.827072,0.011672,-0.273072,128.596679,-0.004317,-0.311671


In [ ]:
print("Shape: ", df_clean.shape)
print("Sum of NaN\n", df_clean.isna().sum().sort_values())

Shape:  (1206, 43)
Sum of NaN
 Open             0
High             0
Low              0
Close            0
Volume           0
SMA_10           0
SMA_20           0
SMA_50           0
EMA_10           0
EMA_20           0
MACD_12_26_9     0
MACDh_12_26_9    0
MACDs_12_26_9    0
RSI_14           0
BBL_5_2.0_2.0    0
BBM_5_2.0_2.0    0
BBU_5_2.0_2.0    0
BBB_5_2.0_2.0    0
BBP_5_2.0_2.0    0
OBV              0
ATR_14           0
STOCHk_14_3_3    0
STOCHd_14_3_3    0
STOCHh_14_3_3    0
MFI_14           0
ADX_14           0
ADXR_14_2        0
DMP_14           0
DMN_14           0
Close_lag_1      0
Return_lag_1     0
Close_lag_2      0
Return_lag_2     0
Close_lag_3      0
Return_lag_3     0
Close_lag_4      0
Return_lag_4     0
Close_lag_5      0
Return_lag_5     0
Volume_change    0
VWAP             0
VWAP_dev         0
Daily_change     0
dtype: int64


# Non-LSTM Based Time-Series Regression Model

# LSTM Based Neural Network Model

In [ ]:
class LSTMModel(nn.Module):
  def init(self, )

# Text-Based Analysis Model

In [ ]:
from bs4 import BeautifulSoup


# Run All Models